In [1]:
import pandas as pd 

In [2]:
clean_df=pd.read_csv("../data/processed_data/cleaned_data.csv")
label_col="Label"
text_col="Text_data"

In [3]:
print("Shape:", clean_df.shape)
print("\nClass distribution:")
print(clean_df[label_col].value_counts())
print("\nClass proportions:")
print(clean_df[label_col].value_counts(normalize=True))
print("\nMissing values:")
print(clean_df.isnull().sum())
print("\nDuplicate texts:")
print(clean_df[text_col].duplicated().sum())

Shape: (6861, 4)

Class distribution:
Label
moderate          3510
not depression    2645
severe             706
Name: count, dtype: int64

Class proportions:
Label
moderate          0.511587
not depression    0.385512
severe            0.102900
Name: proportion, dtype: float64

Missing values:
Unnamed: 0    0
PID           0
Text_data     0
Label         0
dtype: int64

Duplicate texts:
0


In [4]:
class_counts = clean_df[label_col].value_counts()

majority = class_counts.max()
minority = class_counts.min()

imbalance_ratio = majority / minority

print("Majority class:", majority)
print("Minority class:", minority)
print("Imbalance ratio:", imbalance_ratio)

Majority class: 3510
Minority class: 706
Imbalance ratio: 4.971671388101983


In [5]:
text_lengths = clean_df[text_col].str.len()

print(text_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]))

count     6861.000000
mean       842.608075
std       1163.882431
min         12.000000
25%        220.000000
50%        509.000000
75%       1034.000000
95%       2599.000000
99%       5658.000000
max      27432.000000
Name: Text_data, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

train_df, dev_df = train_test_split(
    clean_df,
    test_size=0.20,
    random_state=14,
    stratify=clean_df[label_col]
)

print("Training set:", train_df.shape)
print("Validation set:", dev_df.shape)

print("\nTraining distribution:")
print(train_df[label_col].value_counts(normalize=True))

print("\nValidation distribution:")
print(dev_df[label_col].value_counts(normalize=True))

Training set: (5488, 4)
Validation set: (1373, 4)

Training distribution:
Label
moderate          0.511480
not depression    0.385569
severe            0.102952
Name: proportion, dtype: float64

Validation distribution:
Label
moderate          0.512017
not depression    0.385288
severe            0.102695
Name: proportion, dtype: float64


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=14
    ))
])

X_train = train_df[text_col].astype(str)
y_train = train_df[label_col]

X_dev = dev_df[text_col].astype(str)
y_dev = dev_df[label_col]

baseline_model.fit(X_train, y_train)

print("Baseline training complete!")

Baseline training complete!


In [8]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef
)

y_pred = baseline_model.predict(X_dev)

print("Macro-F1:",
      f1_score(y_dev, y_pred, average="macro"))

print("Weighted-F1:",
      f1_score(y_dev, y_pred, average="weighted"))

print("Balanced Accuracy:",
      balanced_accuracy_score(y_dev, y_pred))

print("MCC:",
      matthews_corrcoef(y_dev, y_pred))

print("\nClassification Report:")
print(classification_report(y_dev, y_pred, zero_division=0))

Macro-F1: 0.4657434474070137
Weighted-F1: 0.5742473487448471
Balanced Accuracy: 0.45866347425840814
MCC: 0.2643144346970316

Classification Report:
                precision    recall  f1-score   support

      moderate       0.59      0.79      0.68       703
not depression       0.61      0.49      0.55       529
        severe       0.78      0.10      0.18       141

      accuracy                           0.60      1373
     macro avg       0.66      0.46      0.47      1373
  weighted avg       0.62      0.60      0.57      1373



In [9]:
train_df.to_csv("../data/processed_data/train.csv")
dev_df.to_csv("../data/processed_data/dev.csv")